<a href="https://colab.research.google.com/github/RadhikaDeshpande1010/-PySpark-RDD-Analytics-Banking-Transactions-Dataset/blob/main/pyspark_rdd_banking_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 PySpark RDD Analytics — Banking Transactions Dataset

**Domain:** Banking Analytics | **Engine:** Apache Spark (RDD API) | **Language:** Python 3

---

## 📋 Dataset Schema

| Field              | Index | Type   | Description                                   |
|--------------------|-------|--------|-----------------------------------------------|
| `transaction_id`   | 0     | int    | Unique transaction identifier                 |
| `customer_name`    | 1     | str    | Customer name                                 |
| `account_type`     | 2     | str    | Account type: `Savings` / `Current`           |
| `transaction_type` | 3     | str    | Type: `Deposit` / `Withdrawal` / `Transfer`   |
| `amount`           | 4     | int    | Transaction amount in ₹ (INR)                 |
| `city`             | 5     | str    | City of the transaction                       |

**Sample record:** `1001,Ravi,Savings,Deposit,5000,Delhi`

---

## 📂 Exercises Overview

| Range    | Topics Covered                                                                 |
|----------|--------------------------------------------------------------------------------|
| Q1 – Q15 | `filter`, `map`, `count`, `reduceByKey`, `sortBy`, `take`, `takeOrdered`       |
| Q16 – Q30| Net balance, currency conversion, composite keys, average per city/type        |
| Q31 – Q50| `flatMap`, `distinct`, `join`, suspicious detection, high/low tagging          |
| Q51 – Q67| Per-customer aggregations, city stats, withdrawal ranking, frequency analysis  |

## ⚙️ Environment Setup

Initialize a local SparkSession and retrieve the SparkContext.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local") \
    .appName("Banking_RDD_Analytics") \
    .getOrCreate()

print(spark)

In [2]:
sc = spark.sparkContext
print(sc)

<SparkContext master=local appName=Banking_RDD_Analytics>


## 📥 Data Ingestion

Load the CSV dataset and parse each record into a list of fields.

In [3]:
# Load raw CSV into an RDD
data_rdd = sc.textFile("/content/sample_data/banking_rdd_dataset.csv") \
              .map(lambda row: row.split(","))
data_rdd.take(5)

[['101', 'Amit', 'Savings', 'Deposit', '5000', 'Patna'],
 ['102', 'Neha', 'Current', 'Withdrawal', '2000', 'Delhi'],
 ['103', 'Ravi', 'Savings', 'Deposit', '7000', 'Mumbai'],
 ['104', 'Priya', 'Savings', 'Transfer', '3000', 'Kolkata'],
 ['105', 'Sohan', 'Current', 'Deposit', '10000', 'Patna']]

---
## 🔄 Core Filters, Maps & Aggregations

Exercises 1–15: `filter`, `map`, `count`, `reduceByKey`, `sortBy`, `take`, `takeOrdered`.

In [4]:
# Q1 — Filter all Savings account transactions
saving_account_transactions = data_rdd.filter(lambda cols: cols[2] == 'Savings')
print(saving_account_transactions.collect())

[['101', 'Amit', 'Savings', 'Deposit', '5000', 'Patna'], ['103', 'Ravi', 'Savings', 'Deposit', '7000', 'Mumbai'], ['104', 'Priya', 'Savings', 'Transfer', '3000', 'Kolkata'], ['106', 'Anjali', 'Savings', 'Withdrawal', '1500', 'Delhi'], ['108', 'Pooja', 'Savings', 'Deposit', '8000', 'Kolkata'], ['109', 'Rahul', 'Savings', 'Withdrawal', '2500', 'Patna'], ['111', 'Karan', 'Savings', 'Transfer', '3500', 'Mumbai'], ['112', 'Nisha', 'Savings', 'Deposit', '9000', 'Kolkata'], ['114', 'Meena', 'Savings', 'Deposit', '6500', 'Delhi'], ['116', 'Simran', 'Savings', 'Withdrawal', '2200', 'Kolkata'], ['117', 'Deepak', 'Savings', 'Deposit', '11000', 'Patna'], ['119', 'Manoj', 'Savings', 'Transfer', '5000', 'Mumbai'], ['120', 'Tina', 'Savings', 'Deposit', '7500', 'Kolkata'], ['121', 'Pooja', 'Savings', 'Deposit', '9000', 'Kolkata'], ['122', 'Rahul', 'Savings', 'Withdrawal', '1500', 'Patna'], ['124', 'Pooja', 'Savings', 'Withdrawal', '5000', 'Kolkata']]


In [5]:
# Q2 — Transactions where amount > 8000 → (customer, type, amount) sorted ascending
transactions_amount = data_rdd \
    .filter(lambda cols: int(cols[4]) > 8000) \
    .map(lambda cols: (cols[1], cols[2], int(cols[4]))) \
    .sortBy(lambda x: x[2], ascending=True)
print(transactions_amount.collect())

[('Nisha', 'Savings', 9000), ('Pooja', 'Savings', 9000), ('Sohan', 'Current', 10000), ('Deepak', 'Savings', 11000), ('Sneha', 'Current', 12000), ('Kavita', 'Current', 13000)]


In [6]:
# Q3 — Extract Deposit transactions → (customer, amount)
deposit_transactions = data_rdd \
    .filter(lambda cols: cols[3] == 'Deposit') \
    .map(lambda cols: (cols[1], int(cols[4])))
print(deposit_transactions.collect())

[('Amit', 5000), ('Ravi', 7000), ('Sohan', 10000), ('Pooja', 8000), ('Sneha', 12000), ('Nisha', 9000), ('Meena', 6500), ('Deepak', 11000), ('Kavita', 13000), ('Tina', 7500), ('Pooja', 9000)]


In [7]:
# Q4 — Filter Patna transactions and display total count
patna_count = data_rdd.filter(lambda cols: cols[5] == 'Patna').count()
print(f'Patna transaction count: {patna_count}')

# Using reduceByKey to confirm
patna_rdk = data_rdd \
    .filter(lambda cols: cols[5] == 'Patna') \
    .map(lambda cols: ('Patna', 1)) \
    .reduceByKey(lambda x, y: x + y)
print(patna_rdk.collect())

Patna transaction count: 7
[('Patna', 7)]


In [8]:
# Q5 — Add ₹500 bonus to every Deposit transaction → (customer, amount+500)
deposit_with_bonus = data_rdd \
    .filter(lambda cols: cols[3] == 'Deposit') \
    .map(lambda cols: (cols[1], int(cols[4]) + 500))
print(deposit_with_bonus.collect())

[('Amit', 5500), ('Ravi', 7500), ('Sohan', 10500), ('Pooja', 8500), ('Sneha', 12500), ('Nisha', 9500), ('Meena', 7000), ('Deepak', 11500), ('Kavita', 13500), ('Tina', 8000), ('Pooja', 9500)]


In [9]:
# Q6 — Total deposit amount across all records using reduceByKey
total_deposit_amount = data_rdd \
    .filter(lambda cols: cols[3] == 'Deposit') \
    .map(lambda cols: (cols[3], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_deposit_amount.collect())

[('Deposit', 98000)]


In [10]:
# Q7 — Count of Withdrawal transactions per account type
withdrawal_count_per_type = data_rdd \
    .filter(lambda cols: cols[3] == 'Withdrawal') \
    .map(lambda cols: (cols[2], 1)) \
    .reduceByKey(lambda x, y: x + y)
print(withdrawal_count_per_type.collect())

[('Current', 2), ('Savings', 5)]


In [11]:
# Q8 — Total transaction amount per account type using reduceByKey
total_transaction = data_rdd \
    .map(lambda cols: (cols[2], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_transaction.collect())

[('Savings', 87200), ('Current', 53500)]


In [12]:
# Q9 — City-wise total transaction amount
city_total_transactions = data_rdd \
    .map(lambda cols: (cols[5], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(city_total_transactions.collect())

[('Patna', 35000), ('Delhi', 35000), ('Mumbai', 27000), ('Kolkata', 43700)]


In [13]:
# Q10 — Net balance per customer (Deposit = +, Withdrawal/Transfer = -)
net_balance = data_rdd \
    .map(lambda cols: (
        cols[1],
        int(cols[4]) if cols[3] == 'Deposit'
        else -int(cols[4]) if cols[3] in ['Withdrawal', 'Transfer']
        else 0
    )) \
    .reduceByKey(lambda x, y: x + y)
print(net_balance.collect())

[('Amit', 5000), ('Neha', -2000), ('Ravi', 7000), ('Priya', -3000), ('Sohan', 10000), ('Anjali', -1500), ('Vikas', -7000), ('Pooja', 12000), ('Rahul', -4000), ('Sneha', 12000), ('Karan', -3500), ('Nisha', 9000), ('Arjun', -5000), ('Meena', 6500), ('Rohit', -4500), ('Simran', -2200), ('Deepak', 11000), ('Kavita', 13000), ('Manoj', -5000), ('Tina', 7500)]


In [14]:
# Q11 — Sort all transactions by amount (descending)
transactions_sorted_desc = data_rdd \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .sortBy(lambda x: x[1], ascending=False)
print(transactions_sorted_desc.collect())

[('Kavita', 13000), ('Sneha', 12000), ('Deepak', 11000), ('Sohan', 10000), ('Nisha', 9000), ('Pooja', 9000), ('Pooja', 8000), ('Tina', 7500), ('Ravi', 7000), ('Meena', 6500), ('Amit', 5000), ('Manoj', 5000), ('Pooja', 5000), ('Rohit', 4500), ('Vikas', 4000), ('Karan', 3500), ('Priya', 3000), ('Arjun', 3000), ('Vikas', 3000), ('Rahul', 2500), ('Simran', 2200), ('Neha', 2000), ('Arjun', 2000), ('Anjali', 1500), ('Rahul', 1500)]


In [15]:
# Q12 — Top 5 highest transactions
top_five_transactions = data_rdd \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(5)
print(top_five_transactions)

[('Kavita', 13000), ('Sneha', 12000), ('Deepak', 11000), ('Sohan', 10000), ('Nisha', 9000)]


In [16]:
# Q13 — Top 3 customers by total transaction amount using takeOrdered
top_customers = data_rdd \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .takeOrdered(3, key=lambda x: -x[1])
print(top_customers)

[('Pooja', 22000), ('Kavita', 13000), ('Sneha', 12000)]


In [17]:
# Q14 — Count occurrences of each transaction type using flatMap + reduceByKey
transaction_count = data_rdd \
    .flatMap(lambda cols: [(cols[3], 1)]) \
    .reduceByKey(lambda x, y: x + y)
print(transaction_count.collect())

[('Deposit', 11), ('Withdrawal', 7), ('Transfer', 7)]


In [18]:
# Q15 — Total transaction count per city using flatMap
city_transaction_count = data_rdd \
    .flatMap(lambda cols: [(cols[5], 1)]) \
    .reduceByKey(lambda x, y: x + y)
print(city_transaction_count.collect())

[('Patna', 7), ('Delhi', 5), ('Mumbai', 6), ('Kolkata', 7)]


---
## 📊 Advanced Aggregations & Composite Keys

Exercises 16–35: top-N, averages, currency conversion, composite keys, multi-condition filters.

In [19]:
# Q16 — Top 3 cities with highest total deposit amount
top_three_cities = data_rdd \
    .filter(lambda cols: cols[3] == 'Deposit') \
    .map(lambda cols: (cols[5], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .takeOrdered(3, key=lambda x: -x[1])
print(top_three_cities)

[('Kolkata', 33500), ('Delhi', 31500), ('Patna', 26000)]


In [20]:
# Q17 — Identify customers who made more than 1 transaction
multi_transaction_customers = data_rdd \
    .map(lambda cols: (cols[1], 1)) \
    .reduceByKey(lambda x, y: x + y) \
    .filter(lambda x: x[1] > 1)
print(multi_transaction_customers.collect())

[('Vikas', 2), ('Pooja', 3), ('Rahul', 2), ('Arjun', 2)]


In [21]:
# Q18 — Average transaction amount per account type
average_transactions = data_rdd \
    .map(lambda cols: (cols[2], (int(cols[4]), 1))) \
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])) \
    .mapValues(lambda x: round(x[0] / x[1], 2))
print(average_transactions.collect())

[('Savings', 5450.0), ('Current', 5944.44)]


In [22]:
# Q19 — Suspicious transactions (amount > 10000) → (customer, city, amount)
suspicious_transactions = data_rdd \
    .filter(lambda cols: int(cols[4]) > 10000) \
    .map(lambda cols: (cols[1], cols[5], int(cols[4])))
print(suspicious_transactions.collect())

[('Sneha', 'Delhi', 12000), ('Deepak', 'Patna', 11000), ('Kavita', 'Delhi', 13000)]


In [23]:
# Q20 — Current account Deposit transactions → (customer, amount, city)
current_deposit_transactions = data_rdd \
    .filter(lambda cols: cols[3] == 'Deposit' and cols[2] == 'Current') \
    .map(lambda cols: (cols[1], int(cols[4]), cols[5]))
print(current_deposit_transactions.collect())

[('Sohan', 10000, 'Patna'), ('Sneha', 12000, 'Delhi'), ('Kavita', 13000, 'Delhi')]


In [24]:
# Q21 — Transactions where amount is between ₹3000 and ₹9000 → (customer, amount)
mid_range_transactions = data_rdd \
    .filter(lambda cols: 3000 < int(cols[4]) <= 9000) \
    .map(lambda cols: (cols[1], int(cols[4])))
print(mid_range_transactions.collect())

[('Amit', 5000), ('Ravi', 7000), ('Vikas', 4000), ('Pooja', 8000), ('Karan', 3500), ('Nisha', 9000), ('Meena', 6500), ('Rohit', 4500), ('Manoj', 5000), ('Tina', 7500), ('Pooja', 9000), ('Pooja', 5000)]


In [25]:
# Q22 — Count Transfer transactions from Mumbai
mumbai_transfer_count = data_rdd \
    .filter(lambda cols: cols[3] == 'Transfer' and cols[5] == 'Mumbai') \
    .count()
print(f'Mumbai Transfer count: {mumbai_transfer_count}')

Mumbai Transfer count: 5


In [26]:
# Q23 — Filter customers whose name starts with 'A' or 'S'
customer_names_as = data_rdd \
    .filter(lambda cols: cols[1].startswith('A') or cols[1].startswith('S')) \
    .map(lambda x: x[1])
print(customer_names_as.collect())

['Amit', 'Sohan', 'Anjali', 'Sneha', 'Arjun', 'Simran', 'Arjun']


In [27]:
# Q24 — Convert transaction amounts from ₹ to USD (1₹ = 0.012 USD)
conversion_rate_inr_to_usd = 0.012

usd_transaction_rdd = data_rdd.map(
    lambda record: (
        record[0], record[1], record[2], record[3],
        round(int(record[4]) * conversion_rate_inr_to_usd, 2),
        record[5]
    )
)
print(usd_transaction_rdd.collect())

[('101', 'Amit', 'Savings', 'Deposit', 60.0, 'Patna'), ('102', 'Neha', 'Current', 'Withdrawal', 24.0, 'Delhi'), ('103', 'Ravi', 'Savings', 'Deposit', 84.0, 'Mumbai'), ('104', 'Priya', 'Savings', 'Transfer', 36.0, 'Kolkata'), ('105', 'Sohan', 'Current', 'Deposit', 120.0, 'Patna'), ('106', 'Anjali', 'Savings', 'Withdrawal', 18.0, 'Delhi'), ('107', 'Vikas', 'Current', 'Transfer', 48.0, 'Mumbai'), ('108', 'Pooja', 'Savings', 'Deposit', 96.0, 'Kolkata'), ('109', 'Rahul', 'Savings', 'Withdrawal', 30.0, 'Patna'), ('110', 'Sneha', 'Current', 'Deposit', 144.0, 'Delhi'), ('111', 'Karan', 'Savings', 'Transfer', 42.0, 'Mumbai'), ('112', 'Nisha', 'Savings', 'Deposit', 108.0, 'Kolkata'), ('113', 'Arjun', 'Current', 'Withdrawal', 36.0, 'Patna'), ('114', 'Meena', 'Savings', 'Deposit', 78.0, 'Delhi'), ('115', 'Rohit', 'Current', 'Transfer', 54.0, 'Mumbai'), ('116', 'Simran', 'Savings', 'Withdrawal', 26.4, 'Kolkata'), ('117', 'Deepak', 'Savings', 'Deposit', 132.0, 'Patna'), ('118', 'Kavita', 'Current', 

In [28]:
# Q25 — Total Deposit amount per account type (Savings vs Current)
total_deposit_by_account = data_rdd \
    .filter(lambda record: record[3] == 'Deposit') \
    .map(lambda cols: (cols[2], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_deposit_by_account.collect())

[('Savings', 63000), ('Current', 35000)]


In [29]:
# Q26 — Total transaction amount per transaction type
total_per_type = data_rdd \
    .map(lambda cols: (cols[3], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_per_type.collect())

[('Deposit', 98000), ('Withdrawal', 17700), ('Transfer', 25000)]


In [30]:
# Q27 — Maximum transaction amount per city using reduceByKey
max_per_city = data_rdd \
    .map(lambda cols: (cols[5], int(cols[4]))) \
    .reduceByKey(lambda x, y: max(x, y))
print(max_per_city.collect())

[('Patna', 11000), ('Delhi', 13000), ('Mumbai', 7000), ('Kolkata', 9000)]


In [31]:
# Q28 — Minimum Withdrawal amount per account type
min_withdrawal = data_rdd \
    .filter(lambda cols: cols[3] == 'Withdrawal') \
    .map(lambda cols: (cols[2], int(cols[4]))) \
    .reduceByKey(lambda x, y: min(x, y))
print(min_withdrawal.collect())

[('Current', 2000), ('Savings', 1500)]


In [32]:
# Q29 — Total transfer amount per customer
total_transfer_per_customer = data_rdd \
    .filter(lambda cols: cols[3] == 'Transfer') \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(total_transfer_per_customer.collect())

[('Priya', 3000), ('Vikas', 7000), ('Karan', 3500), ('Rohit', 4500), ('Manoj', 5000), ('Arjun', 2000)]


In [33]:
# Q30 — Sort customers by total withdrawal amount (descending)
customer_withdrawal_sorted = data_rdd \
    .filter(lambda cols: cols[3] == 'Withdrawal') \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .sortBy(lambda x: x[1], ascending=False)
print(customer_withdrawal_sorted.collect())

[('Pooja', 5000), ('Arjun', 3000), ('Rahul', 2500), ('Simran', 2200), ('Neha', 2000), ('Anjali', 1500), ('Rahul', 1500)]


---
## 🔍 flatMap, Joins, Frequency & City Intelligence

Exercises 36–67: `flatMap`, `distinct`, `join`, deposit-only customers, high-frequency cities, city stats.

In [34]:
# Q31 — Top 3 transactions by amount using takeOrdered
top_three_transactions = data_rdd \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .takeOrdered(3, key=lambda x: -x[1])
print(top_three_transactions)

[('Kavita', 13000), ('Sneha', 12000), ('Deepak', 11000)]


In [35]:
# Q32 — Sort cities by total transaction volume (ascending)
cities_sorted_asc = data_rdd \
    .map(lambda cols: (cols[5], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=True)
print(cities_sorted_asc.collect())

[('Mumbai', 27000), ('Patna', 35000), ('Delhi', 35000), ('Kolkata', 43700)]


In [36]:
# Q33 — Extract all unique cities using flatMap + distinct
unique_cities = data_rdd \
    .flatMap(lambda cols: [cols[5]]) \
    .distinct()
print(unique_cities.collect())

['Patna', 'Delhi', 'Mumbai', 'Kolkata']


In [37]:
# Q34 — Count occurrences of each account type using flatMap + reduceByKey
account_type_count = data_rdd \
    .flatMap(lambda cols: [(cols[2], 1)]) \
    .reduceByKey(lambda x, y: x + y)
print(account_type_count.collect())

[('Savings', 16), ('Current', 9)]


In [38]:
# Q35 — Customers who made both Deposit and Withdrawal transactions
both_deposit_withdrawal = data_rdd \
    .map(lambda cols: (cols[1], {cols[3]})) \
    .reduceByKey(lambda a, b: a.union(b)) \
    .filter(lambda x: 'Deposit' in x[1] and 'Withdrawal' in x[1])
print(both_deposit_withdrawal.collect())

[('Pooja', {'Withdrawal', 'Deposit'})]


In [39]:
# Q36 — Cities where total transaction amount exceeds ₹10,000
high_value_cities = data_rdd \
    .map(lambda cols: (cols[5], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .filter(lambda x: x[1] > 10000)
print(high_value_cities.collect())

[('Patna', 35000), ('Delhi', 35000), ('Mumbai', 27000), ('Kolkata', 43700)]


In [40]:
# Q37 — Savings Withdrawal transactions → (customer, city, amount)
savings_withdrawals = data_rdd \
    .filter(lambda record: record[2] == 'Savings' and record[3] == 'Withdrawal') \
    .map(lambda cols: (cols[1], cols[5], int(cols[4])))
print(savings_withdrawals.collect())

[('Anjali', 'Delhi', 1500), ('Rahul', 'Patna', 2500), ('Simran', 'Kolkata', 2200), ('Rahul', 'Patna', 1500), ('Pooja', 'Kolkata', 5000)]


In [41]:
# Q38 — Count of transactions from Delhi and Kolkata combined
delhi_kolkata_count = data_rdd \
    .filter(lambda cols: cols[5] in ('Delhi', 'Kolkata')) \
    .count()
print(f'Delhi + Kolkata transaction count: {delhi_kolkata_count}')

Delhi + Kolkata transaction count: 12


In [42]:
# Q39 — Increase all amounts by 10%
amount_increased = data_rdd \
    .map(lambda cols: (
        cols[0], cols[1], cols[2], cols[3],
        round(int(cols[4]) * 1.10, 2),
        cols[5]
    ))
print(amount_increased.collect())

[('101', 'Amit', 'Savings', 'Deposit', 5500.0, 'Patna'), ('102', 'Neha', 'Current', 'Withdrawal', 2200.0, 'Delhi'), ('103', 'Ravi', 'Savings', 'Deposit', 7700.0, 'Mumbai'), ('104', 'Priya', 'Savings', 'Transfer', 3300.0, 'Kolkata'), ('105', 'Sohan', 'Current', 'Deposit', 11000.0, 'Patna'), ('106', 'Anjali', 'Savings', 'Withdrawal', 1650.0, 'Delhi'), ('107', 'Vikas', 'Current', 'Transfer', 4400.0, 'Mumbai'), ('108', 'Pooja', 'Savings', 'Deposit', 8800.0, 'Kolkata'), ('109', 'Rahul', 'Savings', 'Withdrawal', 2750.0, 'Patna'), ('110', 'Sneha', 'Current', 'Deposit', 13200.0, 'Delhi'), ('111', 'Karan', 'Savings', 'Transfer', 3850.0, 'Mumbai'), ('112', 'Nisha', 'Savings', 'Deposit', 9900.0, 'Kolkata'), ('113', 'Arjun', 'Current', 'Withdrawal', 3300.0, 'Patna'), ('114', 'Meena', 'Savings', 'Deposit', 7150.0, 'Delhi'), ('115', 'Rohit', 'Current', 'Transfer', 4950.0, 'Mumbai'), ('116', 'Simran', 'Savings', 'Withdrawal', 2420.0, 'Kolkata'), ('117', 'Deepak', 'Savings', 'Deposit', 12100.0, 'Patna

In [43]:
# Q40 — Compute global average transaction amount
total_amount = data_rdd.map(lambda cols: int(cols[4])).sum()
total_count  = data_rdd.count()
avg_amount   = round(total_amount / total_count, 2)
print(f'Global average transaction amount: ₹{avg_amount}')

Global average transaction amount: ₹5628.0


In [44]:
# Q41 — Total amount per (account_type, transaction_type) composite key
amount_per_type_pair = data_rdd \
    .map(lambda cols: ((cols[2], cols[3]), int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y)
print(amount_per_type_pair.collect())

[(('Savings', 'Deposit'), 63000), (('Current', 'Withdrawal'), 5000), (('Savings', 'Transfer'), 11500), (('Current', 'Deposit'), 35000), (('Savings', 'Withdrawal'), 12700), (('Current', 'Transfer'), 13500)]


In [45]:
# Q42 — Average deposit amount per city
# Step 1: filter Deposits → (city, (amount, count=1))
# Step 2: sum totals with reduceByKey
# Step 3: mapValues to compute average
average_deposit_per_city = data_rdd \
    .filter(lambda cols: cols[3] == 'Deposit') \
    .map(lambda cols: (cols[5], (int(cols[4]), 1))) \
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])) \
    .mapValues(lambda x: round(x[0] / x[1], 2))
print(average_deposit_per_city.collect())

[('Patna', 8666.67), ('Mumbai', 7000.0), ('Kolkata', 8375.0), ('Delhi', 10500.0)]


In [46]:
# Q43 — Total unique customers per city
unique_customers_per_city = data_rdd \
    .map(lambda cols: (cols[5], cols[1])) \
    .distinct() \
    .map(lambda x: (x[0], 1)) \
    .reduceByKey(lambda x, y: x + y)
print(unique_customers_per_city.collect())

[('Patna', 5), ('Delhi', 5), ('Mumbai', 5), ('Kolkata', 5)]


In [47]:
# Q44 — Total withdrawal amount per customer, sorted descending
total_withdrawal_per_customer = data_rdd \
    .filter(lambda record: record[3] == 'Withdrawal') \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False)
print(total_withdrawal_per_customer.collect())

[('Pooja', 5000), ('Rahul', 4000), ('Arjun', 3000), ('Simran', 2200), ('Neha', 2000), ('Anjali', 1500)]


In [48]:
# Q45 — Difference between total Deposit and Withdrawal per city using join
total_deposit = data_rdd \
    .filter(lambda record: record[3] == 'Deposit') \
    .map(lambda cols: (cols[5], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y)

total_withdrawal = data_rdd \
    .filter(lambda record: record[3] == 'Withdrawal') \
    .map(lambda cols: (cols[5], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y)

deposit_withdrawal_diff = total_deposit.join(total_withdrawal) \
    .mapValues(lambda x: x[0] - x[1])
print(deposit_withdrawal_diff.collect())

[('Patna', 19000), ('Kolkata', 26300), ('Delhi', 28000)]


In [49]:
# Q46 — Top 2 cities with highest single withdrawal amount
top_two_withdrawal_cities = data_rdd \
    .filter(lambda record: record[3] == 'Withdrawal') \
    .map(lambda cols: (cols[5], int(cols[4]))) \
    .reduceByKey(lambda x, y: max(x, y)) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(2)
print(top_two_withdrawal_cities)

[('Kolkata', 5000), ('Patna', 3000)]


In [50]:
# Q47 — Sort customers by number of transactions (descending)
customer_tx_count_sorted = data_rdd \
    .map(lambda cols: (cols[1], 1)) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False)
print(customer_tx_count_sorted.collect())

[('Pooja', 3), ('Vikas', 2), ('Rahul', 2), ('Arjun', 2), ('Amit', 1), ('Neha', 1), ('Ravi', 1), ('Priya', 1), ('Sohan', 1), ('Anjali', 1), ('Sneha', 1), ('Karan', 1), ('Nisha', 1), ('Meena', 1), ('Rohit', 1), ('Simran', 1), ('Deepak', 1), ('Kavita', 1), ('Manoj', 1), ('Tina', 1)]


In [51]:
# Q48 — Bottom 3 customers by total transaction amount
bottom_3_customers = data_rdd \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=True) \
    .take(3)
print(bottom_3_customers)

[('Anjali', 1500), ('Neha', 2000), ('Simran', 2200)]


In [52]:
# Q49 — (customer, transaction_type) pairs using flatMap
customer_tx_type_pairs = data_rdd \
    .flatMap(lambda cols: [(cols[1], cols[2])])
print(customer_tx_type_pairs.collect())

[('Amit', 'Savings'), ('Neha', 'Current'), ('Ravi', 'Savings'), ('Priya', 'Savings'), ('Sohan', 'Current'), ('Anjali', 'Savings'), ('Vikas', 'Current'), ('Pooja', 'Savings'), ('Rahul', 'Savings'), ('Sneha', 'Current'), ('Karan', 'Savings'), ('Nisha', 'Savings'), ('Arjun', 'Current'), ('Meena', 'Savings'), ('Rohit', 'Current'), ('Simran', 'Savings'), ('Deepak', 'Savings'), ('Kavita', 'Current'), ('Manoj', 'Savings'), ('Tina', 'Savings'), ('Pooja', 'Savings'), ('Rahul', 'Savings'), ('Vikas', 'Current'), ('Pooja', 'Savings'), ('Arjun', 'Current')]


In [53]:
# Q50 — Distinct (city, account_type) pairs using flatMap + distinct
distinct_city_account = data_rdd \
    .flatMap(lambda cols: [(cols[5], cols[2])]) \
    .distinct()
print(distinct_city_account.collect())

[('Patna', 'Savings'), ('Delhi', 'Current'), ('Mumbai', 'Savings'), ('Kolkata', 'Savings'), ('Patna', 'Current'), ('Delhi', 'Savings'), ('Mumbai', 'Current')]


In [54]:
# Q51 — Customers with total deposits > ₹5,000
high_deposit_customers = data_rdd \
    .filter(lambda records: records[3] == 'Deposit' and int(records[4]) > 5000) \
    .map(lambda cols: cols[1])
print(high_deposit_customers.collect())

['Ravi', 'Sohan', 'Pooja', 'Sneha', 'Nisha', 'Meena', 'Deepak', 'Kavita', 'Tina', 'Pooja']


In [55]:
# Q52 — Cities where average transaction amount < ₹5,000
low_avg_cities = data_rdd \
    .map(lambda cols: (cols[5], (int(cols[4]), 1))) \
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])) \
    .mapValues(lambda x: x[0] / x[1]) \
    .filter(lambda x: x[1] < 5000)
print(low_avg_cities.collect())

[('Mumbai', 4500.0)]


In [56]:
# Q53 — Customers who only made Deposit transactions
deposit_only_customers = data_rdd \
    .map(lambda cols: (cols[1], {cols[3]})) \
    .reduceByKey(lambda a, b: a.union(b)) \
    .filter(lambda x: x[1] == {'Deposit'}) \
    .map(lambda x: x[0])
print(deposit_only_customers.collect())

['Amit', 'Ravi', 'Sohan', 'Sneha', 'Nisha', 'Meena', 'Deepak', 'Kavita', 'Tina']


In [57]:
# Q54 — High-frequency cities (more than 5 transactions)
high_frequency_cities = data_rdd \
    .map(lambda cols: (cols[5], 1)) \
    .reduceByKey(lambda x, y: x + y) \
    .filter(lambda x: x[1] > 5)
print(high_frequency_cities.collect())

[('Patna', 7), ('Mumbai', 6), ('Kolkata', 7)]


In [58]:
# Q55 — Customer contributing the highest amount in each city
top_customer_per_city = data_rdd \
    .map(lambda cols: (cols[5], (cols[1], int(cols[4])))) \
    .reduceByKey(lambda x, y: x if x[1] > y[1] else y)
print(top_customer_per_city.collect())

[('Patna', ('Deepak', 11000)), ('Delhi', ('Kavita', 13000)), ('Mumbai', ('Ravi', 7000)), ('Kolkata', ('Pooja', 9000))]


In [59]:
# Q56 — Transactions where amount ends with '000' → (customer, amount)
round_amount_transactions = data_rdd \
    .filter(lambda cols: cols[4].endswith('000')) \
    .map(lambda cols: (cols[1], int(cols[4])))
print(round_amount_transactions.collect())

[('Amit', 5000), ('Neha', 2000), ('Ravi', 7000), ('Priya', 3000), ('Sohan', 10000), ('Vikas', 4000), ('Pooja', 8000), ('Sneha', 12000), ('Nisha', 9000), ('Arjun', 3000), ('Deepak', 11000), ('Kavita', 13000), ('Manoj', 5000), ('Pooja', 9000), ('Vikas', 3000), ('Pooja', 5000), ('Arjun', 2000)]


In [60]:
# Q57 — Transactions where city is not Patna and amount > ₹5,000
non_patna_high_value = data_rdd \
    .filter(lambda cols: cols[5] != 'Patna' and int(cols[4]) > 5000)
print(non_patna_high_value.collect())

[['103', 'Ravi', 'Savings', 'Deposit', '7000', 'Mumbai'], ['108', 'Pooja', 'Savings', 'Deposit', '8000', 'Kolkata'], ['110', 'Sneha', 'Current', 'Deposit', '12000', 'Delhi'], ['112', 'Nisha', 'Savings', 'Deposit', '9000', 'Kolkata'], ['114', 'Meena', 'Savings', 'Deposit', '6500', 'Delhi'], ['118', 'Kavita', 'Current', 'Deposit', '13000', 'Delhi'], ['120', 'Tina', 'Savings', 'Deposit', '7500', 'Kolkata'], ['121', 'Pooja', 'Savings', 'Deposit', '9000', 'Kolkata']]


In [61]:
# Q58 — Customers who made Transfer transactions greater than ₹4,000
high_transfer_customers = data_rdd \
    .filter(lambda cols: cols[3] == 'Transfer' and int(cols[4]) > 4000) \
    .map(lambda x: x[1])
print(high_transfer_customers.collect())

['Rohit', 'Manoj']


In [62]:
# Q59 — Savings account transactions where amount < ₹3,000
low_savings_transactions = data_rdd \
    .filter(lambda cols: cols[2] == 'Savings' and int(cols[4]) < 3000)
print(low_savings_transactions.collect())

[['106', 'Anjali', 'Savings', 'Withdrawal', '1500', 'Delhi'], ['109', 'Rahul', 'Savings', 'Withdrawal', '2500', 'Patna'], ['116', 'Simran', 'Savings', 'Withdrawal', '2200', 'Kolkata'], ['122', 'Rahul', 'Savings', 'Withdrawal', '1500', 'Patna']]


In [63]:
# Q60 — Tag each transaction as 'high' (amount > 8000) or 'low'
tagged_transactions = data_rdd \
    .map(lambda cols: (
        cols[1],
        int(cols[4]),
        'high' if int(cols[4]) > 8000 else 'low'
    ))
print(tagged_transactions.collect())

[('Amit', 5000, 'low'), ('Neha', 2000, 'low'), ('Ravi', 7000, 'low'), ('Priya', 3000, 'low'), ('Sohan', 10000, 'high'), ('Anjali', 1500, 'low'), ('Vikas', 4000, 'low'), ('Pooja', 8000, 'low'), ('Rahul', 2500, 'low'), ('Sneha', 12000, 'high'), ('Karan', 3500, 'low'), ('Nisha', 9000, 'high'), ('Arjun', 3000, 'low'), ('Meena', 6500, 'low'), ('Rohit', 4500, 'low'), ('Simran', 2200, 'low'), ('Deepak', 11000, 'high'), ('Kavita', 13000, 'high'), ('Manoj', 5000, 'low'), ('Tina', 7500, 'low'), ('Pooja', 9000, 'high'), ('Rahul', 1500, 'low'), ('Vikas', 3000, 'low'), ('Pooja', 5000, 'low'), ('Arjun', 2000, 'low')]


In [64]:
# Q61 — Transaction count per (account_type, city) composite key
count_per_account_city = data_rdd \
    .map(lambda cols: ((cols[2], cols[5]), 1)) \
    .reduceByKey(lambda x, y: x + y)
print(count_per_account_city.collect())

[(('Savings', 'Patna'), 4), (('Current', 'Delhi'), 3), (('Savings', 'Mumbai'), 3), (('Savings', 'Kolkata'), 7), (('Current', 'Patna'), 3), (('Savings', 'Delhi'), 2), (('Current', 'Mumbai'), 3)]


In [65]:
# Q62 — Maximum deposit amount per customer
max_deposit_per_customer = data_rdd \
    .filter(lambda records: records[3] == 'Deposit') \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .reduceByKey(lambda x, y: max(x, y))
print(max_deposit_per_customer.collect())

[('Amit', 5000), ('Ravi', 7000), ('Sohan', 10000), ('Pooja', 9000), ('Sneha', 12000), ('Nisha', 9000), ('Meena', 6500), ('Deepak', 11000), ('Kavita', 13000), ('Tina', 7500)]


In [66]:
# Q63 — Transaction count per (city, account_type) composite key
city_account_count = data_rdd \
    .map(lambda cols: ((cols[5], cols[2]), 1)) \
    .reduceByKey(lambda x, y: x + y)
print(city_account_count.collect())

[(('Patna', 'Savings'), 4), (('Delhi', 'Current'), 3), (('Mumbai', 'Savings'), 3), (('Kolkata', 'Savings'), 7), (('Patna', 'Current'), 3), (('Delhi', 'Savings'), 2), (('Mumbai', 'Current'), 3)]


In [67]:
# Q64 — Total and average transaction amount per city
# Step 1: (city, (amount, 1))
# Step 2: reduceByKey → (city, (total, count))
# Step 3: map → (city, total, average)
city_stats = data_rdd \
    .map(lambda cols: (cols[5], (int(cols[4]), 1))) \
    .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1])) \
    .map(lambda x: (x[0], x[1][0], round(x[1][0] / x[1][1], 2)))
print(city_stats.collect())

[('Patna', 35000, 5000.0), ('Delhi', 35000, 7000.0), ('Mumbai', 27000, 4500.0), ('Kolkata', 43700, 6242.86)]


In [68]:
# Q65 — Customers who made both Transfer and Withdrawal transactions
transfer_and_withdrawal = data_rdd \
    .map(lambda cols: (cols[1], {cols[3]})) \
    .reduceByKey(lambda x, y: x.union(y)) \
    .filter(lambda x: 'Transfer' in x[1] and 'Withdrawal' in x[1]) \
    .map(lambda x: x[0])
print(transfer_and_withdrawal.collect())

['Arjun']


In [69]:
# Q66 — Top 5 customers with highest total deposit amount
top_5_depositors = data_rdd \
    .filter(lambda records: records[3] == 'Deposit') \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False) \
    .take(5)
print(top_5_depositors)

[('Pooja', 17000), ('Kavita', 13000), ('Sneha', 12000), ('Deepak', 11000), ('Sohan', 10000)]


In [70]:
# Q67 — Sort cities by number of Withdrawal transactions (descending)
withdrawal_city_ranked = data_rdd \
    .filter(lambda records: records[3] == 'Withdrawal') \
    .map(lambda cols: (cols[5], 1)) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=False)
print(withdrawal_city_ranked.collect())

[('Patna', 3), ('Delhi', 2), ('Kolkata', 2)]


In [71]:
# Q68 — Bottom 3 customers by total transaction amount
lowest_3_customers = data_rdd \
    .map(lambda cols: (cols[1], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .sortBy(lambda x: x[1], ascending=True) \
    .take(3)
print(lowest_3_customers)

[('Anjali', 1500), ('Neha', 2000), ('Simran', 2200)]


In [72]:
# Q69 — Frequency of (customer, city, amount) tuples using flatMap
frequency_data = data_rdd \
    .flatMap(lambda cols: [((cols[1], cols[5], int(cols[4])), 1)]) \
    .reduceByKey(lambda x, y: x + y)
print(frequency_data.collect())

[(('Amit', 'Patna', 5000), 1), (('Neha', 'Delhi', 2000), 1), (('Ravi', 'Mumbai', 7000), 1), (('Priya', 'Kolkata', 3000), 1), (('Sohan', 'Patna', 10000), 1), (('Anjali', 'Delhi', 1500), 1), (('Vikas', 'Mumbai', 4000), 1), (('Pooja', 'Kolkata', 8000), 1), (('Rahul', 'Patna', 2500), 1), (('Sneha', 'Delhi', 12000), 1), (('Karan', 'Mumbai', 3500), 1), (('Nisha', 'Kolkata', 9000), 1), (('Arjun', 'Patna', 3000), 1), (('Meena', 'Delhi', 6500), 1), (('Rohit', 'Mumbai', 4500), 1), (('Simran', 'Kolkata', 2200), 1), (('Deepak', 'Patna', 11000), 1), (('Kavita', 'Delhi', 13000), 1), (('Manoj', 'Mumbai', 5000), 1), (('Tina', 'Kolkata', 7500), 1), (('Pooja', 'Kolkata', 9000), 1), (('Rahul', 'Patna', 1500), 1), (('Vikas', 'Mumbai', 3000), 1), (('Pooja', 'Kolkata', 5000), 1), (('Arjun', 'Patna', 2000), 1)]


In [76]:
# Q70 — Classify each city as 'High Value' or 'Low Value' based on total transaction amount
city_value_classification = data_rdd \
    .map(lambda cols: (cols[5], int(cols[4]))) \
    .reduceByKey(lambda x, y: x + y) \
    .map(lambda x: (x[0], x[1], 'High Value' if x[1] > 20000 else 'Low Value')) \
    .sortBy(lambda x: x[1], ascending=False)
print(city_value_classification.collect())

[('Kolkata', 43700, 'High Value'), ('Patna', 35000, 'High Value'), ('Delhi', 35000, 'High Value'), ('Mumbai', 27000, 'High Value')]
